# 可信井 log-AI 三带分解实验

第一轮高斯二分实验表明，`full - broad` 同时混入了亚米级测井波动、候选薄层结构和仍可被地震正演感知的宽尺度成分。第二轮因此不再直接寻找一个笼统的 high-frequency residual，而是比较：

```text
full log-AI = broad + geological detail + ultrafine
```

两个嵌套的零相位高斯尺度给出严格可重建的三带分解：

```text
broad             = Gaussian(full, broad FWHM)
geological detail = Gaussian(full, fine FWHM) - broad
ultrafine         = full - Gaussian(full, fine FWHM)
```

FWHM 只是平滑尺度，不等同于矩形频带边界。本轮只比较中间带是否具有可辨认的厚度、振幅和非周期地质形态，不训练网络，也不自动选出赢家。


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from scipy import signal
from scipy.ndimage import gaussian_filter1d

repo_root = Path.cwd().resolve()
if not (repo_root / "src").is_dir():
    repo_root = repo_root.parent
if not (repo_root / "src").is_dir():
    raise RuntimeError("Could not locate repository root containing src/.")

src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from cup.physics.numpy_backend import forward_depth, velocity_from_ai
from cup.synthetic.core.signal import finite_support_fir, valid_filter_decimate

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 180,
        "axes.grid": True,
        "grid.alpha": 0.20,
        "font.size": 8.5,
    }
)
print(f"Repository: {repo_root}")

In [ ]:
# 所有实验旋钮集中在本 cell。
SOURCE_RUN_DIR = repo_root / "experiments" / "well_residual_decomposition" / "results" / "20260810_gaussian_scale_space"
RUN_ID = "20260810_three_band_scale_space"
OUTPUT_DIR = repo_root / "experiments" / "well_residual_decomposition" / "results" / RUN_ID

# fine_fwhm_m 控制被隔离到 ultrafine 的最细尺度；
# broad_tuning_fraction 控制 broad 与 geological detail 的分界。
CANDIDATES = {
    "F15_B050": {"fine_fwhm_m": 1.5, "broad_tuning_fraction": 0.50},
    "F30_B050": {"fine_fwhm_m": 3.0, "broad_tuning_fraction": 0.50},
    "F15_B075": {"fine_fwhm_m": 1.5, "broad_tuning_fraction": 0.75},
    "F30_B075": {"fine_fwhm_m": 3.0, "broad_tuning_fraction": 0.75},
}
REFERENCE_CANDIDATE = "F30_B050"
ZOOM_WINDOW_M = 40.0
REVIEW_WINDOW_COUNT = 3
MODEL_GRID_INTERVAL_M = 5.0
FORWARD_OUTPUT_CHUNK_SIZE = 32
MIN_RUN_BROAD_FWHM_MULTIPLE = 1.0

for required in (SOURCE_RUN_DIR / "manifest.json", SOURCE_RUN_DIR / "metrics.csv"):
    if not required.exists():
        raise FileNotFoundError(required)
if REFERENCE_CANDIDATE not in CANDIDATES:
    raise KeyError(REFERENCE_CANDIDATE)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "figures").mkdir(exist_ok=True)
(OUTPUT_DIR / "wells").mkdir(exist_ok=True)
print(f"Source: {SOURCE_RUN_DIR}")
print(f"Output: {OUTPUT_DIR}")

## 1. 复用第一轮的固定输入合同

第二轮读取第一轮已经发布的全频井曲线、TVDSS 轴、层位和冻结正演合同。第一轮的高斯 residual 不参与本轮分解，只作为来源审计。


In [ ]:
source_manifest = json.loads((SOURCE_RUN_DIR / "manifest.json").read_text(encoding="utf-8"))
source_metrics = pd.read_csv(SOURCE_RUN_DIR / "metrics.csv")
source_well_metrics = source_metrics.sort_values(["well_name", "setting"]).groupby("well_name", as_index=True).first()


def resolve_path(value):
    path = Path(str(value))
    return path if path.is_absolute() else repo_root / path


forward_inputs_path = resolve_path(source_manifest["inputs"]["forward_inputs"])
forward_inputs = json.loads(forward_inputs_path.read_text(encoding="utf-8"))
wavelet_path = resolve_path(forward_inputs["wavelet"]["path"])
wavelet_frame = pd.read_csv(wavelet_path)
wavelet_time_s = wavelet_frame["time_s"].to_numpy(dtype=np.float64)
wavelet_amplitude = wavelet_frame["amplitude"].to_numpy(dtype=np.float64)
relation = forward_inputs["ai_velocity_relation"]
AI_VP_A = float(relation["a"])
AI_VP_B = float(relation["b"])

wells = {}
for well_name in source_manifest["trusted_wells"]:
    artifact_path = resolve_path(source_manifest["well_artifacts"][well_name])
    with np.load(artifact_path, allow_pickle=False) as artifact:
        depth = artifact["tvdss_m"].astype(np.float64)
        values = artifact["well_log_ai"].astype(np.float64)
        valid = artifact["well_valid"].astype(bool)
        horizon_names = artifact["horizon_names"].astype(str)
        horizon_depths = artifact["horizon_tvdss_m"].astype(np.float64)
        synthetic_full = artifact["g100_synthetic_full"].astype(np.float64)
    intervals = np.diff(depth)
    dz_m = float(np.median(intervals))
    if np.any(intervals <= 0.0) or not np.allclose(intervals, dz_m, rtol=1e-5, atol=1e-6):
        raise ValueError(f"{well_name}: source axis is not regularly increasing.")
    wells[well_name] = {
        "well_name": well_name,
        "tvdss_m": depth,
        "log_ai": values,
        "valid": valid,
        "dz_m": dz_m,
        "tuning_scale_m": float(source_well_metrics.loc[well_name, "tuning_scale_m"]),
        "tie_corr": float(source_well_metrics.loc[well_name, "tie_corr"]),
        "horizons": dict(zip(horizon_names.tolist(), horizon_depths.tolist())),
        "synthetic_full": synthetic_full,
        "source_artifact": str(artifact_path),
    }

display(
    pd.DataFrame(
        [
            {
                "well_name": well["well_name"],
                "dz_m": well["dz_m"],
                "tuning_scale_m": well["tuning_scale_m"],
                "tie_corr": well["tie_corr"],
                "valid_samples": int(np.count_nonzero(well["valid"])),
            }
            for well in wells.values()
        ]
    )
)

## 2. 三带分解与正演贡献

每个连续有效井段独立滤波，缺口不插值。正演诊断比较三条曲线：`broad`、`broad + geological detail` 和 `full`。因此：

```text
detail forward contribution    = Forward(broad + detail) - Forward(broad)
ultrafine forward contribution = Forward(full) - Forward(broad + detail)
```

它们是同一非线性正演链上的增量诊断，不把 residual 当作独立绝对阻抗正演。


In [ ]:
def finite_runs(mask):
    mask = np.asarray(mask, dtype=bool)
    padded = np.concatenate(([False], mask, [False]))
    changes = np.flatnonzero(padded[1:] != padded[:-1])
    return tuple(slice(int(start), int(stop)) for start, stop in changes.reshape(-1, 2))


def rms(values):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    return float(np.sqrt(np.mean(values**2))) if values.size else np.nan


def smooth_valid_runs(values, valid, dz_m, fwhm_m, minimum_run_m):
    sigma_samples = (float(fwhm_m) / 2.354820045) / float(dz_m)
    minimum_samples = max(3, int(np.ceil(float(minimum_run_m) / float(dz_m))))
    smoothed = np.full(values.shape, np.nan, dtype=np.float64)
    support = np.zeros(values.shape, dtype=bool)
    for run in finite_runs(valid & np.isfinite(values)):
        if run.stop - run.start < minimum_samples:
            continue
        smoothed[run] = gaussian_filter1d(values[run], sigma=sigma_samples, mode="reflect", truncate=4.0)
        support[run] = True
    return smoothed, support


def three_band_decomposition(well, fine_fwhm_m, broad_fwhm_m):
    if not (0.0 < fine_fwhm_m < broad_fwhm_m):
        raise ValueError("Expected 0 < fine FWHM < broad FWHM.")
    minimum_run_m = MIN_RUN_BROAD_FWHM_MULTIPLE * broad_fwhm_m
    fine_smooth, fine_support = smooth_valid_runs(
        well["log_ai"], well["valid"], well["dz_m"], fine_fwhm_m, minimum_run_m
    )
    broad, broad_support = smooth_valid_runs(well["log_ai"], well["valid"], well["dz_m"], broad_fwhm_m, minimum_run_m)
    support = fine_support & broad_support
    detail = np.where(support, fine_smooth - broad, np.nan)
    ultrafine = np.where(support, well["log_ai"] - fine_smooth, np.nan)
    reconstructed = np.where(support, broad + detail + ultrafine, np.nan)
    if not np.any(support):
        raise ValueError(f"{well['well_name']}: no support for three-band decomposition.")
    parity = float(np.max(np.abs(reconstructed[support] - well["log_ai"][support])))
    if parity > 1e-10:
        raise ValueError(f"Three-band reconstruction parity failed: {parity:.6g}")
    return {
        "fine_fwhm_m": float(fine_fwhm_m),
        "broad_fwhm_m": float(broad_fwhm_m),
        "support": support,
        "broad_log_ai": broad,
        "geological_reconstruction_log_ai": fine_smooth,
        "geological_detail": detail,
        "ultrafine": ultrafine,
        "reconstructed_log_ai": reconstructed,
        "reconstruction_max_abs": parity,
    }


def projected_forward_log_ai(depth_m, log_ai, support, highres_interval_m):
    depth_m = np.asarray(depth_m, dtype=np.float64)
    log_ai = np.asarray(log_ai, dtype=np.float64)
    support = np.asarray(support, dtype=bool)
    output = np.full(log_ai.shape, np.nan, dtype=np.float64)
    factor_float = MODEL_GRID_INTERVAL_M / float(highres_interval_m)
    factor = int(round(factor_float))
    if factor < 1 or not np.isclose(factor_float, factor, rtol=0.0, atol=1e-6):
        raise ValueError("High-resolution axis is not nested with the model interval.")
    taps = finite_support_fir(factor)
    for run in finite_runs(support & np.isfinite(log_ai)):
        if run.stop - run.start < 3:
            continue
        model_log_ai, model_support = valid_filter_decimate(log_ai[run], factor=factor, taps=taps)
        model_depth = depth_m[run][::factor]
        model_indices = np.arange(run.start, run.stop, factor, dtype=np.int64)
        if model_log_ai.shape != model_depth.shape or model_depth.shape != model_indices.shape:
            raise ValueError("Projected model arrays have inconsistent shapes.")
        if model_log_ai.size < 2 or not np.any(model_support):
            continue
        local_ai = np.exp(model_log_ai)
        local_vp = velocity_from_ai(local_ai, a=AI_VP_A, b=AI_VP_B)
        synthetic = forward_depth(
            model_log_ai,
            local_vp,
            model_depth,
            wavelet_time_s,
            wavelet_amplitude,
            output_chunk_size=FORWARD_OUTPUT_CHUNK_SIZE,
        )
        output[model_indices[model_support]] = synthetic[model_support]
    return output


def zero_crossing_interval_m(values, support, dz_m):
    intervals = []
    for run in finite_runs(support & np.isfinite(values)):
        local = values[run]
        crossings = np.flatnonzero(np.signbit(local[1:]) != np.signbit(local[:-1]))
        if crossings.size >= 2:
            intervals.extend(np.diff(crossings).astype(np.float64) * float(dz_m))
    return np.asarray(intervals, dtype=np.float64)


def safe_corr(left, right, support):
    selected = support & np.isfinite(left) & np.isfinite(right)
    if np.count_nonzero(selected) < 2:
        return np.nan
    return float(np.corrcoef(left[selected], right[selected])[0, 1])

In [ ]:
all_results = {}
metric_rows = []
artifact_paths = {}

for well_name, well in wells.items():
    candidate_results = {}
    for candidate_name, config in CANDIDATES.items():
        broad_fwhm_m = config["broad_tuning_fraction"] * well["tuning_scale_m"]
        result = three_band_decomposition(
            well,
            fine_fwhm_m=config["fine_fwhm_m"],
            broad_fwhm_m=broad_fwhm_m,
        )
        synthetic_geological = projected_forward_log_ai(
            well["tvdss_m"],
            result["geological_reconstruction_log_ai"],
            result["support"],
            well["dz_m"],
        )
        synthetic_broad = projected_forward_log_ai(
            well["tvdss_m"], result["broad_log_ai"], result["support"], well["dz_m"]
        )
        result["synthetic_full"] = well["synthetic_full"]
        result["synthetic_geological"] = synthetic_geological
        result["synthetic_broad"] = synthetic_broad
        result["synthetic_detail_contribution"] = synthetic_geological - synthetic_broad
        result["synthetic_ultrafine_contribution"] = well["synthetic_full"] - synthetic_geological

        common = (
            result["support"]
            & np.isfinite(well["synthetic_full"])
            & np.isfinite(synthetic_geological)
            & np.isfinite(synthetic_broad)
        )
        forward_status = "ok" if np.count_nonzero(common) >= 2 else "unavailable_no_common_support"
        full_forward_rms = rms(well["synthetic_full"][common])
        detail_intervals = zero_crossing_interval_m(result["geological_detail"], result["support"], well["dz_m"])
        ultrafine_intervals = zero_crossing_interval_m(result["ultrafine"], result["support"], well["dz_m"])
        metric_rows.append(
            {
                "well_name": well_name,
                "candidate": candidate_name,
                "fine_fwhm_m": result["fine_fwhm_m"],
                "broad_fwhm_m": result["broad_fwhm_m"],
                "broad_tuning_fraction": config["broad_tuning_fraction"],
                "supported_samples": int(np.count_nonzero(result["support"])),
                "reconstruction_max_abs": result["reconstruction_max_abs"],
                "geological_detail_rms": rms(result["geological_detail"][result["support"]]),
                "ultrafine_rms": rms(result["ultrafine"][result["support"]]),
                "detail_to_ultrafine_rms_ratio": (
                    rms(result["geological_detail"][result["support"]])
                    / max(rms(result["ultrafine"][result["support"]]), 1e-12)
                ),
                "detail_zero_crossing_median_m": (
                    float(np.median(detail_intervals)) if detail_intervals.size else np.nan
                ),
                "ultrafine_zero_crossing_median_m": (
                    float(np.median(ultrafine_intervals)) if ultrafine_intervals.size else np.nan
                ),
                "forward_status": forward_status,
                "forward_supported_samples": int(np.count_nonzero(common)),
                "geological_forward_corr": safe_corr(well["synthetic_full"], synthetic_geological, common),
                "broad_forward_corr": safe_corr(well["synthetic_full"], synthetic_broad, common),
                "detail_forward_rms_ratio": (
                    rms(result["synthetic_detail_contribution"][common]) / max(full_forward_rms, 1e-12)
                    if forward_status == "ok"
                    else np.nan
                ),
                "ultrafine_forward_rms_ratio": (
                    rms(result["synthetic_ultrafine_contribution"][common]) / max(full_forward_rms, 1e-12)
                    if forward_status == "ok"
                    else np.nan
                ),
            }
        )
        candidate_results[candidate_name] = result

    all_results[well_name] = candidate_results
    artifact_path = OUTPUT_DIR / "wells" / f"{well_name}.npz"
    payload = {
        "tvdss_m": well["tvdss_m"],
        "well_log_ai": well["log_ai"],
        "well_valid": well["valid"],
        "horizon_names": np.asarray(list(well["horizons"].keys()), dtype="U64"),
        "horizon_tvdss_m": np.asarray(list(well["horizons"].values()), dtype=np.float64),
    }
    for candidate_name, result in candidate_results.items():
        prefix = candidate_name.lower()
        for key in (
            "support",
            "broad_log_ai",
            "geological_reconstruction_log_ai",
            "geological_detail",
            "ultrafine",
            "synthetic_full",
            "synthetic_geological",
            "synthetic_broad",
            "synthetic_detail_contribution",
            "synthetic_ultrafine_contribution",
        ):
            payload[f"{prefix}_{key}"] = result[key]
        payload[f"{prefix}_fine_fwhm_m"] = np.asarray(result["fine_fwhm_m"])
        payload[f"{prefix}_broad_fwhm_m"] = np.asarray(result["broad_fwhm_m"])
    np.savez_compressed(artifact_path, **payload)
    artifact_paths[well_name] = str(artifact_path)
    print(f"completed decomposition: {well_name}")

metrics = pd.DataFrame(metric_rows).sort_values(["well_name", "candidate"])
metrics_path = OUTPUT_DIR / "metrics.csv"
metrics.to_csv(metrics_path, index=False)
display(metrics)

## 3. 短窗口图件

每口井自动选择一个中间带能量较高、且有效支持完整的约 40 m 窗口。四组候选严格使用同一个窗口和同一组横轴范围。另为参考候选输出三个分散窗口，避免只观察一个最漂亮区间。


In [ ]:
CANDIDATE_COLORS = {
    "F15_B050": "#1f77b4",
    "F30_B050": "#ff7f0e",
    "F15_B075": "#2ca02c",
    "F30_B075": "#d62728",
}


def target_limits(well):
    finite_depth = well["tvdss_m"][well["valid"]]
    horizons = np.asarray(list(well["horizons"].values()), dtype=np.float64)
    lower = max(float(np.min(finite_depth)), float(np.min(horizons)))
    upper = min(float(np.max(finite_depth)), float(np.max(horizons)))
    if upper <= lower:
        raise ValueError(f"{well['well_name']}: invalid target interval.")
    return lower, upper


def strongest_window(well, detail, support, lower, upper, window_m):
    depth = well["tvdss_m"]
    eligible = support & (depth >= lower) & (depth <= upper) & np.isfinite(detail)
    window_samples = max(5, int(round(float(window_m) / well["dz_m"])))
    kernel = np.ones(window_samples, dtype=np.float64)
    energy = signal.convolve(np.where(eligible, detail**2, 0.0), kernel, mode="same")
    count = signal.convolve(eligible.astype(np.float64), kernel, mode="same")
    energy[count < 0.90 * window_samples] = -np.inf
    if not np.any(np.isfinite(energy)):
        center = 0.5 * (lower + upper)
    else:
        center = float(depth[int(np.argmax(energy))])
    half = 0.5 * min(float(window_m), upper - lower)
    center = float(np.clip(center, lower + half, upper - half)) if upper - lower >= 2 * half else 0.5 * (lower + upper)
    return center - half, center + half


def review_windows(well, reference_result):
    lower, upper = target_limits(well)
    edges = np.linspace(lower, upper, REVIEW_WINDOW_COUNT + 1)
    windows = []
    for start, stop in zip(edges[:-1], edges[1:]):
        local_width = min(ZOOM_WINDOW_M, stop - start)
        windows.append(
            strongest_window(
                well,
                reference_result["geological_detail"],
                reference_result["support"],
                float(start),
                float(stop),
                local_width,
            )
        )
    return tuple(windows)


def fill_component(axis, values, depth, color, label):
    axis.plot(values, depth, color=color, linewidth=0.8, label=label)
    axis.fill_betweenx(depth, 0.0, values, where=values >= 0.0, color="#c44e52", alpha=0.28)
    axis.fill_betweenx(depth, 0.0, values, where=values < 0.0, color="#4c72b0", alpha=0.28)
    axis.axvline(0.0, color="0.45", linewidth=0.6)


def plot_candidate_comparison(well, results):
    well_dir = OUTPUT_DIR / "figures" / well["well_name"]
    well_dir.mkdir(parents=True, exist_ok=True)
    reference = results[REFERENCE_CANDIDATE]
    lower, upper = target_limits(well)
    zoom_lower, zoom_upper = strongest_window(
        well,
        reference["geological_detail"],
        reference["support"],
        lower,
        upper,
        ZOOM_WINDOW_M,
    )
    depth = well["tvdss_m"]
    view = (depth >= zoom_lower) & (depth <= zoom_upper)
    detail_limit = float(
        np.percentile(
            np.abs(np.concatenate([item["geological_detail"][view & item["support"]] for item in results.values()])),
            99.0,
        )
    )
    ultrafine_limit = float(
        np.percentile(
            np.abs(np.concatenate([item["ultrafine"][view & item["support"]] for item in results.values()])), 99.0
        )
    )
    full_limits = np.percentile(well["log_ai"][view & well["valid"]], [1.0, 99.0])

    fig, axes = plt.subplots(len(results), 4, figsize=(13.5, 10.5), sharey=True, constrained_layout=True)
    for row, (name, item) in enumerate(results.items()):
        color = CANDIDATE_COLORS[name]
        axes[row, 0].plot(well["log_ai"], depth, color="0.20", linewidth=0.65, label="full")
        axes[row, 0].plot(
            item["geological_reconstruction_log_ai"], depth, color=color, linewidth=1.0, label="broad + detail"
        )
        axes[row, 0].set_xlim(*full_limits)
        axes[row, 0].set_ylabel(f"{name}\nTVDSS (m)")
        axes[row, 0].legend(fontsize=6.5, loc="best")

        fill_component(axes[row, 1], item["geological_detail"], depth, color, "geological detail")
        axes[row, 1].set_xlim(-detail_limit, detail_limit)
        fill_component(axes[row, 2], item["ultrafine"], depth, "0.35", "ultrafine")
        axes[row, 2].set_xlim(-ultrafine_limit, ultrafine_limit)

        forward_view = (
            view
            & np.isfinite(item["synthetic_full"])
            & np.isfinite(item["synthetic_geological"])
            & np.isfinite(item["synthetic_broad"])
        )
        if np.count_nonzero(forward_view) >= 2:
            axes[row, 3].plot(
                item["synthetic_full"][forward_view],
                depth[forward_view],
                color="black",
                marker="o",
                markersize=2.2,
                linewidth=0.9,
                label="full",
            )
            axes[row, 3].plot(
                item["synthetic_geological"][forward_view],
                depth[forward_view],
                color=color,
                marker="o",
                markersize=2.0,
                linewidth=0.9,
                label="broad + detail",
            )
            axes[row, 3].plot(
                item["synthetic_broad"][forward_view],
                depth[forward_view],
                color="0.60",
                marker="o",
                markersize=1.8,
                linewidth=0.8,
                label="broad",
            )
            axes[row, 3].legend(fontsize=6.2, loc="best")
        else:
            axes[row, 3].text(
                0.5,
                0.5,
                "no common forward support",
                transform=axes[row, 3].transAxes,
                ha="center",
                va="center",
                fontsize=7,
            )
        axes[row, 3].axvline(0.0, color="0.45", linewidth=0.6)

        for column in range(4):
            axes[row, column].set_ylim(zoom_upper, zoom_lower)
        axes[row, 0].text(
            0.02,
            0.02,
            f"fine={item['fine_fwhm_m']:.1f} m | broad={item['broad_fwhm_m']:.1f} m",
            transform=axes[row, 0].transAxes,
            fontsize=6.5,
            va="bottom",
        )
    for column, title in enumerate(("full vs retained", "geological detail", "ultrafine", "forward contribution")):
        axes[0, column].set_title(title)
    path = well_dir / "candidate_comparison.png"
    fig.suptitle(f"{well['well_name']} | common {zoom_upper - zoom_lower:.1f} m high-activity window")
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path, (zoom_lower, zoom_upper)


def plot_reference_windows(well, result):
    well_dir = OUTPUT_DIR / "figures" / well["well_name"]
    windows = review_windows(well, result)
    depth = well["tvdss_m"]
    fig, axes = plt.subplots(len(windows), 3, figsize=(10.5, 8.5), constrained_layout=True)
    for row, (lower, upper) in enumerate(windows):
        view = (depth >= lower) & (depth <= upper)
        axes[row, 0].plot(well["log_ai"], depth, color="0.20", linewidth=0.65, label="full")
        axes[row, 0].plot(
            result["geological_reconstruction_log_ai"], depth, color="#ff7f0e", linewidth=1.0, label="retained"
        )
        axes[row, 0].set_xlim(*np.percentile(well["log_ai"][view & well["valid"]], [1.0, 99.0]))
        axes[row, 0].legend(fontsize=6.5)
        fill_component(axes[row, 1], result["geological_detail"], depth, "#ff7f0e", "detail")
        fill_component(axes[row, 2], result["ultrafine"], depth, "0.35", "ultrafine")
        for column in range(3):
            axes[row, column].set_ylim(upper, lower)
        axes[row, 0].set_ylabel(f"TVDSS (m)\nwindow {row + 1}")
    for column, title in enumerate(("full vs retained", "geological detail", "ultrafine")):
        axes[0, column].set_title(title)
    path = well_dir / f"{REFERENCE_CANDIDATE.lower()}_three_windows.png"
    fig.suptitle(f"{well['well_name']} | {REFERENCE_CANDIDATE} | three target-zone windows")
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path, windows


figure_manifest = {}
for well_name, well in wells.items():
    comparison_path, main_window = plot_candidate_comparison(well, all_results[well_name])
    windows_path, windows = plot_reference_windows(well, all_results[well_name][REFERENCE_CANDIDATE])
    figure_manifest[well_name] = {
        "candidate_comparison": str(comparison_path),
        "reference_three_windows": str(windows_path),
        "main_window_tvdss_m": list(main_window),
        "review_windows_tvdss_m": [list(item) for item in windows],
    }
    print(f"completed figures: {well_name}")

In [ ]:
# 跨井汇总：每口井使用自己的共同短窗口；列间只改变候选尺度。
fig, axes = plt.subplots(
    len(wells),
    len(CANDIDATES),
    figsize=(12.5, 2.8 * len(wells)),
    constrained_layout=True,
    squeeze=False,
)
for row, (well_name, well) in enumerate(wells.items()):
    depth = well["tvdss_m"]
    lower, upper = figure_manifest[well_name]["main_window_tvdss_m"]
    view = (depth >= lower) & (depth <= upper)
    common_values = np.concatenate(
        [result["geological_detail"][view & result["support"]] for result in all_results[well_name].values()]
    )
    limit = float(np.percentile(np.abs(common_values), 99.0))
    for column, (candidate_name, result) in enumerate(all_results[well_name].items()):
        fill_component(
            axes[row, column],
            result["geological_detail"],
            depth,
            CANDIDATE_COLORS[candidate_name],
            candidate_name,
        )
        axes[row, column].set_xlim(-limit, limit)
        axes[row, column].set_ylim(upper, lower)
        if row == 0:
            axes[row, column].set_title(candidate_name)
        if column == 0:
            axes[row, column].set_ylabel(f"{well_name}\nTVDSS (m)")
atlas_path = OUTPUT_DIR / "figures" / "trusted_wells_geological_detail_atlas.png"
fig.suptitle("Three-band geological-detail candidates | common short windows")
fig.savefig(atlas_path, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(atlas_path)))
print(atlas_path)

## 4. 发布结果与人工评价模板

本轮不设置自动优胜分数。请先看跨井 atlas，再逐井看 `candidate_comparison.png`，最后检查参考候选的三个分散窗口。重点判断中间带是否保留可变厚度和振幅，以及超细带是否主要像测井毛刺。


In [ ]:
review = metrics[["well_name", "candidate"]].copy()
for column in (
    "geological_detail_meaningful",
    "variable_thickness_and_amplitude",
    "fixed_wavelength_striping",
    "broad_trend_leakage",
    "ultrafine_looks_like_noise",
    "preferred_for_next_round",
    "notes",
):
    review[column] = ""
review_path = OUTPUT_DIR / "human_review.csv"
review.to_csv(review_path, index=False)

manifest = {
    "schema": "well_log_three_band_scale_space_v1",
    "run_id": RUN_ID,
    "status": ("completed_with_warnings" if metrics["forward_status"].ne("ok").any() else "completed"),
    "sample_domain": "depth",
    "sample_unit": "m",
    "depth_basis": "tvdss",
    "source_manifest": str(SOURCE_RUN_DIR / "manifest.json"),
    "source_schema": source_manifest["schema"],
    "trusted_wells": list(wells),
    "candidates": CANDIDATES,
    "reference_candidate_for_multiwindow_review": REFERENCE_CANDIDATE,
    "zoom_window_m": ZOOM_WINDOW_M,
    "well_artifacts": artifact_paths,
    "figures": figure_manifest,
    "atlas": str(atlas_path),
    "metrics": str(metrics_path),
    "human_review": str(review_path),
}
manifest_path = OUTPUT_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

display(
    metrics[
        [
            "well_name",
            "candidate",
            "fine_fwhm_m",
            "broad_fwhm_m",
            "geological_detail_rms",
            "ultrafine_rms",
            "detail_zero_crossing_median_m",
            "ultrafine_zero_crossing_median_m",
            "detail_forward_rms_ratio",
            "ultrafine_forward_rms_ratio",
            "geological_forward_corr",
        ]
    ]
)
print(f"Manifest: {manifest_path}")
print(f"Human review template: {review_path}")

## 5. 建议人工检查顺序

1. 查看 `trusted_wells_geological_detail_atlas.png`：哪一组中间带跨井仍能保留不同形态，而不是统一条纹。
2. 查看逐井 `candidate_comparison.png`：比较中间带和超细带的分工，并观察去掉超细成分后正演波形是否基本保持。
3. 查看 `f30_b050_three_windows.png`：确认结论不是由单个高活动窗口造成。
4. 在 `human_review.csv` 记录评价。

如果四组中间带仍然都像规则滤波纹理，或者 ultrafine 对正演贡献仍然很强，本轮应作为三带尺度空间的负结果，不继续通过增加更多相似 FWHM 来寻找漂亮图件。
